In [41]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import networkx as nx
import ast

In [ ]:
#'Qwen/Qwen2.5-0.5B-Instruct','meta-llama/Llama-3.2-1B-Instruct','mistralai/Mistral-7B-Instruct-v0.2'

In [2]:
def hypothesis_test(statistic, p_value, alpha):
    print(f'P-value: {p_value}')
    print(f'Alpha: {alpha}')
    if p_value < alpha:
        print("Result: Reject the null hypothesis.")
        print("The distributions of the two samples are significantly different.")
        print('different distributions')
    else:
        print("Result: Cannot reject the null hypothesis.")
        print("There is not enough evidence to say the distributions differ.")
        print('not different distributions')

In [6]:
def plot_hist_diff(data_A, data_B, differences):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # PLOT A: Raw Distributions (The Shape Check)
    # We want the mass to be on the RIGHT (near 1.0)
    ax1.hist(data_A, bins=30, alpha=0.6, label='Dataset A', color='blue')
    ax1.hist(data_B, bins=30, alpha=0.6, label='Dataset B', color='red')
    ax1.axvline(np.median(data_A), color='blue', linestyle='--', linewidth=1, label=f'Median A: {np.median(data_A):.2f}')
    ax1.axvline(np.median(data_B), color='red', linestyle='--', linewidth=1, label=f'Median B: {np.median(data_B):.2f}')
    ax1.set_title('Comparison of Match Distributions')
    ax1.set_xlabel('%_matched')
    ax1.set_ylabel('Count')
    ax1.legend()

    # PLOT B: Paired Differences (The Wilcoxon Visualizer)
    # Positive values mean A > B. Negative means B > A.
    median_diff = np.median(differences)
    ax2.hist(differences, bins=30, color='purple', alpha=0.7)
    ax2.axvline(0, color='black', linewidth=2, linestyle='-') # The Tie Line
    ax2.axvline(median_diff, color='red', linestyle='--', linewidth=2, label=f'Median Diff: {median_diff:.3f}')
    ax2.set_title(f'Histogram of Differences (A - B)')
    ax2.set_xlabel('Difference (A - B)')
    ax2.set_ylabel('Count of Pairs')
    ax2.legend()

    plt.tight_layout()
    plt.show()

In [4]:
k = 5
alpha = 0.05

In [ ]:
def run_quantitative(model_name, inference_type, k):
    df_base=pd.DataFrame.from_dict(pd.read_pickle(open(f'{inference_type}/sentence_encoder_skill_matching_results_{model_name.split("/")[-1]}.pkl','rb'))).transpose()
    df_top_2=pd.DataFrame.from_dict(pd.read_pickle(open(f'{inference_type}/sentence_encoder_skill_matching_results_{model_name.split("/")[-1]}_top{k}_weighted_2.pkl','rb'))).transpose()
    df_top_4=pd.DataFrame.from_dict(pd.read_pickle(open(f'{inference_type}/sentence_encoder_skill_matching_results_{model_name.split("/")[-1]}_top{k}_weighted_4.pkl','rb'))).transpose()
    df_top_8=pd.DataFrame.from_dict(pd.read_pickle(open(f'{inference_type}/sentence_encoder_skill_matching_results_{model_name.split("/")[-1]}_top{k}_weighted_8.pkl','rb'))).transpose()

    common_ids = list(set(df_base.index).intersection(set(df_top_2.index)).intersection(set(df_top_4.index)).intersection(set(df_top_8.index)))

    df_base = df_base.loc[common_ids]
    df_top_2 = df_top_2.loc[common_ids]
    df_top_4 = df_top_4.loc[common_ids]
    df_top_8 = df_top_8.loc[common_ids]

    df_base['%_matched_skills'] = df_base['%_matched_skills']/100
    df_top_2['%_matched_skills'] = df_top_2['%_matched_skills']/100
    df_top_4['%_matched_skills'] = df_top_4['%_matched_skills']/100
    df_top_8['%_matched_skills'] = df_top_8['%_matched_skills']/100

    print('Mean Coverage Scores:')
    print('Base:', df_base['%_matched_skills'].mean())
    print('Top 2:', df_top_2['%_matched_skills'].mean())
    print('Top 4:', df_top_4['%_matched_skills'].mean())
    print('Top 8:', df_top_8['%_matched_skills'].mean())

    statistic_1, p_value_1 = stats.wilcoxon(list(df_base['%_matched_skills'].values),list(df_top_2['%_matched_skills'].values), alternative='two-sided')
    median_diff_1 = np.median(np.array(df_base['%_matched_skills']) - np.array(df_top_2['%_matched_skills']))
    plot_hist_diff(df_base['%_matched_skills'], df_top_2['%_matched_skills'], (np.array(df_base['%_matched_skills']) - np.array(df_top_2['%_matched_skills'])))

    statistic_2, p_value_2 = stats.wilcoxon(list(df_base['%_matched_skills'].values),list(df_top_4['%_matched_skills'].values), alternative='two-sided')
    median_diff_2 = np.median(np.array(df_base['%_matched_skills']) - np.array(df_top_4['%_matched_skills']))
    plot_hist_diff(df_base['%_matched_skills'], df_top_4['%_matched_skills'], (np.array(df_base['%_matched_skills']) - np.array(df_top_4['%_matched_skills'])))

    statistic_3, p_value_3 = stats.wilcoxon(list(df_base['%_matched_skills'].values),list(df_top_8['%_matched_skills'].values), alternative='two-sided')
    median_diff_3 = np.median(np.array(df_base['%_matched_skills']) - np.array(df_top_8['%_matched_skills']))
    plot_hist_diff(df_base['%_matched_skills'], df_top_8['%_matched_skills'], (np.array(df_base['%_matched_skills']) - np.array(df_top_8['%_matched_skills'])))
    
    statistic_4, p_value_4 = stats.wilcoxon(list(df_top_2['%_matched_skills'].values),list(df_top_4['%_matched_skills'].values), alternative='two-sided')
    median_diff_4 = np.median(np.array(df_top_2['%_matched_skills']) - np.array(df_top_4['%_matched_skills']))

    statistic_5, p_value_5 = stats.wilcoxon(list(df_top_2['%_matched_skills'].values),list(df_top_8['%_matched_skills'].values), alternative='two-sided')
    median_diff_5 = np.median(np.array(df_top_2['%_matched_skills']) - np.array(df_top_8['%_matched_skills']))

    statistic_6, p_value_6 = stats.wilcoxon(list(df_top_4['%_matched_skills'].values),list(df_top_8['%_matched_skills'].values), alternative='two-sided')
    median_diff_6 = np.median(np.array(df_top_4['%_matched_skills']) - np.array(df_top_8['%_matched_skills']))
    
    for i in range(1,7):
        print(f'\nHypothesis test {i} for the top{k}:')
        hypothesis_test(eval(f'statistic_{i}'), eval(f'p_value_{i}'), alpha)
        print('-------------')
        if eval(f'p_value_{i}') > 0.05:
            print("Result: No significant difference. The datasets are statistically tied.")
        else:
            if eval(f'median_diff_{i}') > 0:
                print("Result: Dataset A is the winner (statistically closer to 1) - value: ",  str(eval(f'median_diff_{i}'))) 
            else:
                print("Result: Dataset B is the winner (statistically closer to 1) - value: ",  str(eval(f'median_diff_{i}')))
        print('-------------')
    print('\nCalculating RMSE for the base model:')
    print(float(np.sqrt(((1 - df_base['%_matched_skills']) ** 2).mean())))
    print(float(np.sqrt(((1 - df_base['%_matched_skills']) ** 2).std())))

    print(f'\nCalculating RMSE for the top{k} model  - times 2:')
    print(float(np.sqrt(((1 - df_top_2['%_matched_skills']) ** 2).mean())))
    print(float(np.sqrt(((1 - df_top_2['%_matched_skills']) ** 2).std())))

    print(f'\nCalculating RMSE for the top{k} model  - times 4:')
    print(float(np.sqrt(((1 - df_top_4['%_matched_skills']) ** 2).mean())))
    print(float(np.sqrt(((1 - df_top_4['%_matched_skills']) ** 2).std())))

    print(f'\nCalculating RMSE for the top{k} model  - times 8:')
    print(float(np.sqrt(((1 - df_top_8['%_matched_skills']) ** 2).mean())))
    print(float(np.sqrt(((1 - df_top_8['%_matched_skills']) ** 2).std())))
    
    print('\nDistribution skews')
    print('Base 2',stats.skew(list(df_base['%_matched_skills'].values)))
    print('Top 2',stats.skew(list(df_top_2['%_matched_skills'].values)))
    print('Top 4',stats.skew(list(df_top_4['%_matched_skills'].values)))
    print('Top 8',stats.skew(list(df_top_8['%_matched_skills'].values)))

    print('\nWasserstein Distance')
    print('Base vs Top 2:', stats.wasserstein_distance(list(df_base['%_matched_skills'].values), list(df_top_2['%_matched_skills'].values)))
    print('Base vs Top 4:', stats.wasserstein_distance(list(df_base['%_matched_skills'].values), list(df_top_4['%_matched_skills'].values)))
    print('Base vs Top 8:', stats.wasserstein_distance(list(df_base['%_matched_skills'].values), list(df_top_8['%_matched_skills'].values)))
    print('Top 2 vs Top 4:', stats.wasserstein_distance(list(df_top_2['%_matched_skills'].values), list(df_top_4['%_matched_skills'].values)))
    print('Top 2 vs Top 8:', stats.wasserstein_distance(list(df_top_2['%_matched_skills'].values), list(df_top_8['%_matched_skills'].values)))
    print('Top 4 vs Top 8:', stats.wasserstein_distance(list(df_top_4['%_matched_skills'].values), list(df_top_8['%_matched_skills'].values)))

In [5]:
def plot_KS_test(data1, data2):
    plt.figure(figsize=(10, 6))
    plt.hist(data1, bins=30, density=True, histtype='step', cumulative=True, label='df1', linewidth=1.5)
    plt.hist(data2, bins=30, density=True, histtype='step', cumulative=True, label='df2', linewidth=1.5)
    plt.legend(loc='upper left')
    plt.title('Cumulative Step Histograms (Visualizing K-S Test)')
    plt.show()

In [82]:
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
inference_type = 'qwen_non_det'

In [ ]:
run_quantitative(model_name, inference_type, 5)
run_quantitative(model_name, inference_type, 10)
run_quantitative(model_name, inference_type, 15)
run_quantitative(model_name, inference_type, 20)

In [8]:
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
inference_type = 'qwen_det'

In [ ]:
run_quantitative(model_name, inference_type, 5)
run_quantitative(model_name, inference_type, 10)
run_quantitative(model_name, inference_type, 15)
run_quantitative(model_name, inference_type, 20)

In [ ]:
model_name = 'meta-llama/Llama-3.2-1B-Instruct'
inference_type = 'llama_non_det'

run_quantitative(model_name, inference_type, 5)
run_quantitative(model_name, inference_type, 10)
run_quantitative(model_name, inference_type, 15)
run_quantitative(model_name, inference_type, 20)

In [56]:
model_name = 'meta-llama/Llama-3.2-1B-Instruct'
inference_type = 'llama_det'

run_quantitative(model_name, inference_type, 5)
run_quantitative(model_name, inference_type, 10)
run_quantitative(model_name, inference_type, 15)
run_quantitative(model_name, inference_type, 20)

In [ ]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.2'
inference_type = 'mistral_non_det'

run_quantitative(model_name, inference_type, 5)
run_quantitative(model_name, inference_type, 10)
run_quantitative(model_name, inference_type, 15)
run_quantitative(model_name, inference_type, 20)

In [ ]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.2'
inference_type = 'mistral_det'

run_quantitative(model_name, inference_type, 5)
run_quantitative(model_name, inference_type, 10)
run_quantitative(model_name, inference_type, 15)
run_quantitative(model_name, inference_type, 20)